In [1]:
import numpy as np
from modules.utils import generate_orthonormal_states

dim = 4
m = 3
states = generate_orthonormal_states(dim, m)
print("Orthogonal states shape:", states.shape)  # (m, dim)
assert np.allclose(states @ states.T.conj(), np.eye(m)), "state vectors are not orthogonal"

Orthogonal states shape: (3, 4)


In [2]:
import pennylane as qml

dev = qml.device('default.qubit', wires=2)

@qml.qnode(dev)
def circuit(state=None):
    qml.StatePrep(state, wires=range(2))
    return qml.expval(qml.Z(0)), qml.state()

state = states[0]
expval, final_state = circuit(state)
print("Expected value:", expval)

Expected value: 0.4147931533302824


In [3]:
from torch.autograd import Variable
import torch

import importlib
from modules import QGAN, Discriminator, MINE  # 초기 import
importlib.reload(QGAN)  # 모듈 갱신
importlib.reload(Discriminator)  # 모듈 갱신
importlib.reload(MINE)  # 모듈 갱신

n_layers = 5
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)
discriminator_initial_params = Variable(torch.tensor(np.random.normal(-np.pi/2 , np.pi/2, (n_layers, n_qubits, 3))), requires_grad=True)
discriminator = QGAN.QDiscriminator(n_qubits, n_layers, discriminator_initial_params, dev)

In [4]:
discriminator.forward([torch.tensor(states[0])])

tensor([0.5391], dtype=torch.float64, grad_fn=<StackBackward0>)